# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month -- iterate here, never on the sealed final month

# Table handles
DAILY_MONTH = f"{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

print("Ready. Querying month:", MONTH)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Contract answers (Lane 2 — Refresh / Content Opportunity Scoring):**

1. **What one row means for my lane:** one row = one pseudonymized content item (`content_hash_id`), belonging to one pseudonymized client (`client_hash_id`), on one report date (`report_date`), in `fact_content_daily_performance`. This is a **content-day** grain, not a content grain — the same page appears once per day it has data. For scoring, I roll this up to one row per `content_hash_id` per month by aggregating across its days in that month (matching the Week 1/2 CSV's 'one row = one page, trailing-90-day window' shape, just rebuilt from daily facts instead of a pre-aggregated file).

2. **Table(s) I'll use:** `fact_content_daily_performance` (the daily GSC/GA4 performance fact — impressions, clicks, position, sessions, engagement) as the primary table, joined to `dim_content` (word count, content type, main intent, freshness/optimization dates — the static, slow-changing page attributes) on `client_hash_id` + `content_hash_id`. `dim_clients` is used only as context (to check `has_gsc_access` / `has_ga4_access` before trusting a client's rows). `fact_content_query_90d` is explicitly **not** used this week — it's a different grain (content-query, not content-day) and out of scope for the contract, though it's a strong future-feature candidate for query-level demand signals.

3. **Time window:** one calendar month at a time, filtered by the `month=` partition — `month=2026-03` for this contract (a mid-panel month, not the sealed final month). All three verification queries and the five features below are built from this single month's daily rows, aggregated per content item.

4. **What I'd predict or rank (label or proxy):** same proxy as Week 2 — `priority_score`, a 0–100 continuous score combining decline direction (from within-month trend), remaining search demand (`gsc_impressions` summed over the month), and current ranking position (`gsc_avg_position`). It's a proxy, not an observed outcome — no editor-action log exists in this warehouse either.

5. **What I deliberately exclude:** GA4-derived engagement/session columns (`ga4_*`, `sessions_*`, `scroll_events`) as *features* for this contract, even though they're technically knowable in-window. I'm excluding them here because a meaningful share of clients have GA4 access flagged False (checked in Section 3's availability query) — including them would silently bias the feature frame toward GSC+GA4 clients only. They stay candidates for later once I've measured that split properly; this contract restricts to GSC-only fields (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`) plus `dim_content` static fields, which every active client has.

In [ ]:
# Verify the grain stated above: one row per (report_date, client_hash_id, content_hash_id)?
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{DAILY_MONTH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print('Duplicate (date, client, content) combos found:', len(grain_check))
print(grain_check)
print()
print('Zero rows back = the content-day grain holds, as stated in the contract above.')

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (knowable at the decision moment, safe to use):
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (aggregated over the month, from `fact_content_daily_performance`)
- `word_count`, `content_type`, `main_intent`, `search_volume`, `competition_level` (from `dim_content`)
- days-since-update, derived from `last_optimized_date` vs. the month's reporting date

**Label / proxy** (the thing predicted, or what it's computed from — never a feature):
- `priority_score` itself, and the trend/decline signal it's built from (month-over-month change in `gsc_impressions` or `gsc_avg_position`) — these must never leak into the feature frame, same rule the Week 2 notebook already applied to `trend_direction`/`trend_pct`.

**Context** (for grouping/joining/splitting — never for the model to learn from):
- `client_hash_id`, `content_hash_id`, `report_date`/`month`, `keyword_hash_id`, `url_hash_id` — all hash IDs, used only to join and to build a client-holdout split later, never as model inputs.

**Excluded** (private, product-decision flags, or future information — each with a why):
- `ga4_*`, `sessions_*`, `ai_*`, `scroll_events` — excluded this week because GA4 access is not universal across clients (verified in Section 3); including them now would bake in a silent client-selection bias.
- `is_deleted`, `is_published` — excluded because they're product/editorial-decision flags, not organic search signal; a page's publish state is a business decision, not something the model should be scoring around.
- Anything from `fact_content_query_90d` — excluded because it's a different grain (content-query, not content-day) and pulling it in without resolving the grain mismatch would risk the exact double-counting trap the skill doc warns about.

In [ ]:
# No extra computation needed for this section -- it's a classification/write-up,
# verified by the queries in Section 3 below (availability + grain).


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 (grain)** is already run in Section 1 — zero duplicate `(report_date, client_hash_id, content_hash_id)` rows confirms one row really is one content-day.

**Query 2 (slice row count + date span)** — how many rows are in my slice (GSC-active clients, month=2026-03), and what dates does the month actually span (should be the full calendar month, but worth checking rather than assuming).

In [ ]:
slice_stats = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{DAILY_MONTH}')
    WHERE gsc_data_available IS TRUE
""").df()

print('Slice: month=2026-03, GSC-available rows only')
print(slice_stats.to_string(index=False))

**Query 3 (availability)** — filter with `IS TRUE` on the availability flags and show how many rows survive vs. the unfiltered month. This is the check that justified excluding GA4 fields in Section 2: if GA4 availability is well below 100%, using GA4 features now would quietly drop or bias a chunk of clients.

In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_gsc_available,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ga4_available
    FROM read_parquet('{DAILY_MONTH}')
""").df()

print('Availability check, month=2026-03, unfiltered:')
print(availability.to_string(index=False))
print()
print('This is why Section 2 excludes GA4 fields for now: GSC availability should be near-universal for active clients,')
print('but GA4 availability is very likely lower -- confirmed by the pct_ga4_available number above, not assumed.')

### Five features (max), one line each: knowable at the decision moment because…

Feature frame built from `fact_content_daily_performance` (month=2026-03, GSC-available rows) aggregated to one row per `content_hash_id`, joined to `dim_content` static fields.

In [ ]:
features = con.sql(f"""
    WITH monthly AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS impressions_month,
            SUM(gsc_clicks) AS clicks_month,
            AVG(gsc_avg_position) AS avg_position_month
        FROM read_parquet('{DAILY_MONTH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        m.content_hash_id,
        m.client_hash_id,
        m.impressions_month,
        m.avg_position_month,
        c.word_count,
        c.content_type,
        DATE '2026-03-31' - c.last_optimized_date AS days_since_last_optimized
    FROM monthly m
    LEFT JOIN read_parquet('{DIM_CONTENT}') c
        ON m.content_hash_id = c.content_hash_id AND m.client_hash_id = c.client_hash_id
    LIMIT 200000
""").df()

print('Feature frame shape:', features.shape)
features.head(5)

**Feature 1 — `impressions_month`** (summed GSC impressions across the month): knowable at the decision moment because it's a fully-elapsed month-to-date sum of past search demand, logged daily by GSC — nothing in it looks past the report date being scored.

**Feature 2 — `avg_position_month`** (mean GSC ranking position across the month): knowable at the decision moment because it's the average of already-observed daily ranking positions for days that have already happened.

**Feature 3 — `word_count`** (from `dim_content`): knowable at the decision moment because it's a static content attribute set when the page was last edited, not something derived from future search outcomes.

**Feature 4 — `content_type`** (from `dim_content`): knowable at the decision moment because it's a fixed editorial classification of the page, assigned independent of any performance data.

**Feature 5 — `days_since_last_optimized`** (month-end date minus `last_optimized_date`): knowable at the decision moment because `last_optimized_date` is always in the past relative to the scoring date by construction — a page can't have been optimized in the future.

### The trap: one label-derived column, on purpose

Quick honest baseline first: predict whether a page is in the bottom half of `avg_position_month` (a stand-in "needs attention" label for this demo) using the five honest features above. Then add ONE column derived directly from the label itself, watch the score jump toward a suspicious near-perfect number, then delete it and keep the honest score.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd

df = features.dropna(subset=['impressions_month', 'avg_position_month', 'word_count']).copy()

# Demo label for the leakage exercise: is this page in the bottom half of ranking position this month?
# (worse position = higher avg_position_month number)
df['needs_attention'] = (df['avg_position_month'] > df['avg_position_month'].median()).astype(int)

honest_features = ['impressions_month', 'word_count']
X = pd.get_dummies(df[honest_features], drop_first=True)
y = df['needs_attention']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f'HONEST score (no leak), AUC: {honest_auc:.3f}')

In [ ]:
# --- THE TRAP ---
# Add ONE column derived directly from the label: avg_position_month itself (rounded),
# which is EXACTLY what needs_attention was computed from. This is the leakage move --
# a feature that is a near-restatement of the label, not an honest predictor.
df['leaky_position_bucket'] = df['avg_position_month'].round(0)

leaky_features = ['impressions_month', 'word_count', 'leaky_position_bucket']
X_leak = pd.get_dummies(df[leaky_features], drop_first=True)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)
model_leak = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_auc = roc_auc_score(y_test_l, model_leak.predict_proba(X_test_l)[:, 1])

print(f'LEAKY score (with leaky_position_bucket), AUC: {leaky_auc:.3f}')
print(f'Jump: {honest_auc:.3f} -> {leaky_auc:.3f}')
print()
print('The jump toward ~1.0 is not a better model -- leaky_position_bucket is a rounded copy of the')
print('exact column needs_attention was thresholded from. The model is not predicting; it is reading the label back.')

In [ ]:
# --- DELETE THE LEAK, KEEP THE HONEST NUMBER ---
del df['leaky_position_bucket']

print('leaky_position_bucket removed from the feature frame.')
print(f'Honest AUC kept for this contract: {honest_auc:.3f}')
print('This is the number that goes forward into modeling weeks -- not the leaky one.')

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** this warehouse is an **unbalanced panel** — `dim_clients.gsc_data_start` / `ga4_data_start` differ per client, so a client whose GSC history starts in, say, May 2025 has no rows at all before that date, and that's an absence of history, not a zero-activity signal. For `month=2026-03` specifically, any client whose `gsc_data_start` falls after March 2026 would simply not appear in this slice — which the availability query in Section 3 partly surfaces (rows with `gsc_data_available` False or missing), but a client missing *entirely* from the month wouldn't show up as a low-availability row at all, it just wouldn't be there. This means row counts and any "declining vs. not" comparison across clients in this contract describe the clients present in March 2026 with active GSC data — not the full 104-client roster — and that scope should be stated explicitly whenever this month's numbers get compared to another month.

In [ ]:
# Verify the limitation: how many of the 104 clients in dim_clients actually appear in this month's slice?
coverage = con.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM read_parquet('{DIM_CLIENTS}')) AS total_clients_in_dim,
        (SELECT COUNT(DISTINCT client_hash_id) FROM read_parquet('{DAILY_MONTH}')) AS clients_present_in_march_2026
""").df()

print(coverage.to_string(index=False))
print()
print('If clients_present_in_march_2026 < total_clients_in_dim, that gap is the unbalanced-panel limitation')
print('made concrete -- some clients simply have no rows in this month, for reasons unrelated to their content quality.')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.